# Lesson 23A - Instructor solutions

Attempt the learner notebook first. These reference implementations explain the choices and
check the underlying data as well as the chart structure. Worked-example setup is repeated so
this notebook runs independently. Some checks still need a human to inspect readability and interpretation.

## 1. Start with a question and inspect the table

A chart maps data to visual properties: position, length, colour, or shape. Start with a
question rather than a favourite chart. Measurements need units; category labels need names.
The loader below reads packaged data locally. Do not replace it with `sns.load_dataset`,
which can fetch data from the internet. Rename columns once so later code stays readable.

Predict the row count and per-species counts before running. Inspect missing values before
drawing a distribution: an omitted row can change the story.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import plotly.express as px
from sklearn.datasets import load_iris

iris = load_iris(as_frame=True)
feature_names = ["sepal_length_cm","sepal_width_cm","petal_length_cm","petal_width_cm"]
flowers = iris.data.copy()
flowers.columns = feature_names
species_order = list(iris.target_names)
flowers["species"] = iris.target.map(dict(enumerate(species_order)))
flowers["sample_id"] = np.arange(len(flowers))
print(flowers.head(3))
counts = flowers["species"].value_counts().reindex(species_order)
print(counts.to_dict())
assert flowers.shape == (150,6)
assert counts.tolist() == [50,50,50]
assert flowers[feature_names].isna().sum().sum() == 0

## 2. Choose a chart and a plotting library

| Question | Chart | Watch for |
| --- | --- | --- |
| How many flowers belong to each species? | Bar chart | Start a count axis at zero |
| How are petal lengths distributed? | Histogram | Bin edges change apparent shape |
| How do petal lengths vary by species? | Box plot | A box summarizes; it does not show every point |
| Are two measurements associated? | Scatter plot | Association is not causation; avoid joining unordered samples |
| How do four numeric features correlate? | Heatmap | Correlation can mix group differences and within-group relationships |
| How does an ordered measurement change? | Line plot | Sort a real order/time variable; sample IDs are not time |

Matplotlib gives precise control over figures, axes, labels, and static exports.
Seaborn builds statistical views on Matplotlib and works naturally with tidy DataFrames.
Plotly creates interactive figures for hover, zoom, legend toggling, and HTML sharing.
Each has a place: a publication figure, a quick statistical comparison, and an exploratory
interactive view can use the same data with different presentation needs.

## 3. Matplotlib: Figure, Axes, and a count chart

A Figure is the full canvas; an Axes is one plotting region. `plt.subplots` returns both.
Prefer `ax.method(...)` so the target axes is explicit, especially with several panels.
Use `set` for titles/axis labels, `bar` for category counts, and `show` to display.
Closing a figure after display prevents accumulating open figures on repeated runs.

Predict the three bar heights. Change one filtered subset and recompute the counts; do not
change only a label. A zero baseline makes bar lengths comparable.

In [ ]:
with plt.rc_context({"font.size":10}):
    fig, ax = plt.subplots(figsize=(6,3.5),layout="constrained")
    bars = ax.bar(counts.index,counts.to_numpy(),color="steelblue")
    ax.bar_label(bars,padding=3)
    ax.set(title="Iris sample counts",xlabel="Species",ylabel="Flowers",ylim=(0,60))
    ax.grid(axis="y",alpha=0.2)
    assert [bar.get_height() for bar in bars] == [50,50,50]
    plt.show()
    plt.close(fig)

## 4. Histograms: bins and counts

A histogram divides a numeric range into intervals and counts observations in each one.
Unlike a category bar chart, its intervals have numerical meaning. NumPy histograms include
the last edge in the last bin; intermediate bins exclude their upper edge.
Use identical edges when comparing groups. Few bins can hide structure; too many can exaggerate noise.
Density is a different scale from counts; density area totals one, not bar heights.

Predict the sum of bin counts, then change bins from 10 to 5 and explain what changes and what remains.

In [ ]:
values = flowers["petal_length_cm"].to_numpy()
bin_edges = np.linspace(values.min(),values.max(),11)  # Ten shared intervals.
fig, ax = plt.subplots(figsize=(6,3.5),layout="constrained")
heights, edges, patches = ax.hist(values,bins=bin_edges,color="steelblue",edgecolor="white")
ax.set(title="Distribution of petal length",xlabel="Petal length (cm)",ylabel="Flowers")
assert int(heights.sum()) == 150 and len(patches) == 10
plt.show()
plt.close(fig)

## 5. Scatter plots, categories, and readable encodings

A scatter plot places one point at each pair of measurements. Use a different marker as
well as colour for species, and include a legend; colour alone may not be accessible.
Transparency reduces overlap. Keep the category order consistent across charts.
Do not connect these observations with a line: their row order is not a chronological sequence.

Predict which species occupies the short-petal region. Swap the x/y features and describe
which visual patterns survive. Labels must change with the columns.

In [ ]:
palette = dict(zip(species_order,["#0072B2","#D55E00","#009E73"],strict=True))
markers = dict(zip(species_order,["o","s","^"],strict=True))
fig, ax = plt.subplots(figsize=(6,4),layout="constrained")
for species in species_order:
    group = flowers.loc[flowers["species"] == species]
    ax.scatter(group["petal_length_cm"],group["petal_width_cm"],
               label=species,color=palette[species],marker=markers[species],alpha=0.7,s=28)
ax.set(title="Petal measurements by species",xlabel="Petal length (cm)",ylabel="Petal width (cm)")
ax.legend(title="Species")
assert sum(len(collection.get_offsets()) for collection in ax.collections) == 150
plt.show()
plt.close(fig)

## 6. Seaborn: compare distributions with a box plot

Seaborn accepts the DataFrame and column names rather than requiring you to split groups
manually. A box spans the middle 50% (quartiles) with a median line. Default whiskers extend
to observations within 1.5 IQR of the quartiles; points beyond are flagged, not automatically errors.
Use a table alongside the chart to verify the summarized values. Counts matter when groups
have unequal sample sizes. All three groups here have 50 observations.

Predict the shortest typical petal length. Compare a median-based box with the mean-based
bar chart in section 7: they answer different questions.

In [ ]:
with sns.axes_style("whitegrid"):
    fig, ax = plt.subplots(figsize=(6,3.5),layout="constrained")
    sns.boxplot(data=flowers,x="species",y="petal_length_cm",order=species_order,
                color="lightsteelblue",ax=ax)
    ax.set(title="Petal length distributions",xlabel="Species",ylabel="Petal length (cm)")
    plt.show()
    plt.close(fig)
summary = flowers.groupby("species")["petal_length_cm"].agg(["count","mean","median"])
summary = summary.reindex(species_order)
print(summary.round(3))
assert summary["count"].tolist() == [50,50,50]
assert summary["median"].idxmin() == "setosa"

## 7. Subplots and explicit statistical aggregation

A two-panel comparison is easier to read when each panel has a clear question. Seaborn's
histplot can separate species with hue. barplot estimates a mean by default and normally
adds uncertainty intervals. Here `errorbar=None` deliberately omits uncertainty; the title
states this is a descriptive mean, not a confidence claim. Use `errorbar="sd"` for observed
spread; a bootstrap confidence interval estimates uncertainty in a mean and means something else.

Predict the number of bars and total histogram counts. Change the histogram bin count and
explain why this does not change the group means.

In [ ]:
with sns.axes_style("whitegrid"):
    fig, axes = plt.subplots(1,2,figsize=(10,3.5),layout="constrained")
    sns.histplot(data=flowers,x="petal_length_cm",hue="species",hue_order=species_order,
                 palette=palette,bins=10,multiple="stack",ax=axes[0])
    axes[0].set(title="Distribution by species",xlabel="Petal length (cm)",ylabel="Flowers")
    sns.barplot(data=flowers,x="species",y="petal_length_cm",order=species_order,
                estimator="mean",errorbar=None,color="steelblue",ax=axes[1])
    axes[1].set(title="Sample mean (no uncertainty shown)",xlabel="Species",ylabel="Mean petal length (cm)")
    axes[1].set_ylim(bottom=0)
    assert len(axes[1].patches) == 3
    np.testing.assert_allclose([bar.get_height() for bar in axes[1].patches],summary["mean"])
    plt.show()
    plt.close(fig)

## 8. Correlation heatmaps and their limits

Pearson correlation measures linear association on a scale from -1 to 1. Compute it on
the four numeric measurements, excluding species codes and sample IDs. A diverging palette
centred at zero and fixed limits make cells comparable. Labels and annotations help avoid
depending only on colour.

Correlations across all species can differ from correlations within one species. A high
correlation does not prove causation, feature importance, or model performance. This plot
is exploratory; if you choose model features based on data, use training data for that decision.

In [ ]:
correlation = flowers[feature_names].corr()
fig, ax = plt.subplots(figsize=(7,5),layout="constrained")
sns.heatmap(correlation,annot=True,fmt=".2f",vmin=-1,vmax=1,center=0,
            cmap="vlag",square=True,ax=ax,cbar_kws={"label":"Pearson r"})
ax.set_title("Linear associations among Iris measurements")
assert correlation.shape == (4,4)
np.testing.assert_allclose(np.diag(correlation),1)
np.testing.assert_allclose(correlation,correlation.T)
plt.show()
plt.close(fig)

## 9. Plotly Express: hover, zoom, and legend filtering

Plotly Express builds an interactive figure from a DataFrame. Hover shows details; dragging
zooms; double-click resets; clicking a legend entry toggles a group. `color` and `symbol`
encode species consistently. Hover IDs identify observations without connecting them as time.
Axis labels should expose units rather than raw internal column names.

The `plotly_mimetype` renderer sends compact figure data to a compatible notebook frontend.
No cloud upload or CDN is required. Some frontends need renderer support: if a blank area appears,
use the self-contained local HTML export in section 11, or try `renderer="notebook"`, which
embeds JavaScript and produces larger notebook outputs. Avoid connected/CDN renderers offline.
Prediction/check: there should be three traces and 150 points in total.

In [ ]:
labels = {"petal_length_cm":"Petal length (cm)","petal_width_cm":"Petal width (cm)",
          "sepal_length_cm":"Sepal length (cm)","sepal_width_cm":"Sepal width (cm)",
          "species":"Species","sample_id":"Sample ID"}
interactive = px.scatter(flowers,x="petal_length_cm",y="petal_width_cm",color="species",
    symbol="species",category_orders={"species":species_order},color_discrete_map=palette,
    hover_data=["sample_id","sepal_length_cm"],labels=labels,
    title="Explore petal measurements",opacity=0.75)
interactive.update_layout(template="plotly_white",height=400)
assert len(interactive.data) == 3
assert sum(len(trace.x) for trace in interactive.data) == 150
interactive.show(renderer="plotly_mimetype")

## 10. Facets make group comparisons explicit

A facet creates a small panel for each category. Use a shared numerical range when direct
comparison matters; otherwise autoscaling can make very different groups look similar.
Plotly's facet_col splits the same scatter question into three panels. This is a good way
to compare within-group patterns with section 9's combined view.

Change the plotted measurements to sepal length/width and explain why the species overlap
may look different. Different views reveal different aspects of the same data.

In [ ]:
faceted = px.scatter(flowers,x="sepal_length_cm",y="sepal_width_cm",color="species",
    facet_col="species",category_orders={"species":species_order},color_discrete_map=palette,
    labels=labels,hover_data=["sample_id"],title="Sepal measurements: one panel per species")
faceted.update_xaxes(range=[4,8.2])
faceted.update_yaxes(range=[1.8,4.6])
faceted.update_layout(template="plotly_white",height=350,showlegend=False)
assert len(faceted.data) == 3
assert sum(len(trace.x) for trace in faceted.data) == 150
faceted.show(renderer="plotly_mimetype")

## 11. Export static and interactive figures locally

Save a Matplotlib figure with `fig.savefig` before closing it. PNG suits documents; SVG
stays sharp when resized. Plotly `write_html(..., include_plotlyjs=True)` embeds the JavaScript
needed for offline use. It produces a few-megabyte HTML file even for a small dataset; avoid
printing or embedding the full HTML as notebook text. Using `"cdn"` instead makes the file
smaller but requires internet. Plotly static image export needs an additional toolchain, so
we use Matplotlib for PNG/SVG and do not require Kaleido or a browser installation.

The test below uses a temporary directory and removes exports afterwards. To keep a report,
choose a deliberate new local path using the optional commented line; the notebook does not
automatically overwrite a project file. Sharing HTML also shares the embedded plotted data.

In [ ]:
from pathlib import Path
from tempfile import TemporaryDirectory

with TemporaryDirectory() as folder:
    folder = Path(folder)
    fig, ax = plt.subplots(figsize=(6,3.5),layout="constrained")
    ax.bar(counts.index,counts.to_numpy(),color="steelblue")
    ax.set(title="Iris sample counts",xlabel="Species",ylabel="Flowers",ylim=(0,60))
    png_path, svg_path = folder/"counts.png", folder/"counts.svg"
    fig.savefig(png_path,dpi=120,bbox_inches="tight")
    fig.savefig(svg_path,bbox_inches="tight")
    plt.close(fig)
    assert png_path.read_bytes().startswith(b"\x89PNG\r\n\x1a\n")
    assert "<svg" in svg_path.read_text(encoding="utf-8")
    html_path = folder/"petals.html"
    interactive.write_html(html_path,include_plotlyjs=True,full_html=True,auto_open=False)
    assert "Plotly.newPlot" in html_path.read_text(encoding="utf-8")
    print("PNG, SVG, and self-contained HTML exports checked; temporary files removed on exit.")
# Optional deliberate export after choosing a new path:
# interactive.write_html("my_iris_exploration.html",include_plotlyjs=True,auto_open=False)

## 12. Common mistakes and debugging

| Symptom | What to inspect |
| --- | --- |
| Plot is blank | Selected columns, filtering count, show call, and the frontend's Plotly renderer |
| Chart says 150 but shows fewer observations | Missing values, nonfinite data, clipping, and overlapping points |
| Bar heights do not match counts | Seaborn barplot estimates a mean; use explicit counts or countplot for frequency |
| Mean chart hides unequal groups | Report sample sizes and inspect a distribution, not just an average |
| One subplot is overwritten | Pass ax= explicitly for Seaborn, and use the intended axes object |
| Repeated plots accumulate | Close each Matplotlib figure after display/export |
| Category colours change between charts | Pass a stable order and category-to-colour mapping |
| A line implies a trend | Ask whether x has a meaningful order; a sample ID is not time |
| Interactive plot works online only | Avoid connected/CDN renderers; export HTML with embedded JavaScript |

Use informative titles and units, restrained colour, readable labels, and shapes/annotations
alongside colour. Start bar axes at zero. State filters and aggregation rules. A truncated scatter
axis can be appropriate for detail when it is clearly labeled; hiding it can mislead.
Inspect raw/grouped values before trusting the visual. Do not remove outliers merely because
they are inconvenient, and do not infer causation from an attractive scatter plot.

## Exercise 1. Count categories with Matplotlib

An explicit count Series makes aggregation visible. Equal bar heights reveal the balanced
sample, without claiming the real population has equal species frequencies.

In [ ]:
practice_counts = flowers["species"].value_counts().reindex(species_order)
assert practice_counts.tolist() == [50,50,50] and practice_counts.sum() == 150
fig, ax = plt.subplots(figsize=(6,3.5),layout="constrained")
bars = ax.bar(practice_counts.index,practice_counts.to_numpy(),color="steelblue")
ax.bar_label(bars,padding=3)
ax.set(title="Species in the Iris teaching sample",xlabel="Species",ylabel="Flowers",ylim=(0,60))
assert [bar.get_height() for bar in bars] == [50,50,50]
plt.show()
plt.close(fig)

## Exercise 2. Compare two histogram bin choices

Equal ranges isolate the effect of binning. Counts, rather than density, let us verify the
number of included observations directly.

In [ ]:
practice_values = flowers["sepal_length_cm"].to_numpy()
fig, axes = plt.subplots(1,2,figsize=(9,3.5),sharex=True,sharey=True,layout="constrained")
for ax, number in zip(axes,[5,15],strict=True):
    edges = np.linspace(practice_values.min(),practice_values.max(),number+1)
    counts_here, _, bars = ax.hist(practice_values,bins=edges,color="steelblue",edgecolor="white")
    assert counts_here.sum() == 150 and len(bars) == number
    ax.set(title=f"{number} bins",xlabel="Sepal length (cm)",ylabel="Flowers")
plt.show()
plt.close(fig)

## Exercise 3. Compare group spread with Seaborn

The table verifies the summary behind the picture. Whisker rules flag observations relative
to a distribution; they do not establish a measurement error.

In [ ]:
practice_summary = flowers.groupby("species")["petal_width_cm"].agg(["median","count"])
practice_summary = practice_summary.reindex(species_order)
assert practice_summary["count"].tolist() == [50,50,50]
assert practice_summary["median"].idxmin() == "setosa"
assert practice_summary["median"].idxmax() == "virginica"
fig, ax = plt.subplots(figsize=(6,3.5),layout="constrained")
sns.boxplot(data=flowers,x="species",y="petal_width_cm",order=species_order,color="lightsteelblue",ax=ax)
ax.set(title="Petal width spread",xlabel="Species",ylabel="Petal width (cm)")
print(practice_summary)
plt.show()
plt.close(fig)

## Exercise 4. Audit a correlation heatmap

Excluding IDs and encoded classes prevents treating category identifiers as measurements.
Pooled and within-group correlations answer different questions; group structure affects the pooled view.

In [ ]:
practice_corr = flowers[feature_names].corr()
assert practice_corr.shape == (4,4)
np.testing.assert_allclose(practice_corr,practice_corr.T)
np.testing.assert_allclose(np.diag(practice_corr),1)
practice_setosa = flowers.loc[flowers["species"] == "setosa",feature_names].corr()
assert practice_corr.loc["petal_length_cm","petal_width_cm"] > practice_setosa.loc["petal_length_cm","petal_width_cm"]
fig, ax = plt.subplots(figsize=(7,5),layout="constrained")
sns.heatmap(practice_corr,vmin=-1,vmax=1,center=0,cmap="vlag",annot=True,fmt=".2f",ax=ax)
ax.set_title("Iris measurement correlations")
plt.show()
plt.close(fig)

## Exercise 5. Build a filtered interactive scatter

Filtering the table explicitly makes the chart's population clear. Each species is one trace,
and the title states the subset rather than implying the full dataset is shown.

In [ ]:
practice_subset = flowers.loc[flowers["species"].isin(["versicolor","virginica"])].copy()
practice_fig = px.scatter(practice_subset,x="petal_length_cm",y="petal_width_cm",
    color="species",symbol="species",color_discrete_map=palette,
    category_orders={"species":["versicolor","virginica"]},hover_data=["sample_id"],
    labels=labels,title="Petal overlap: versicolor and virginica only")
assert len(practice_fig.data) == 2
assert sum(len(trace.x) for trace in practice_fig.data) == 100
assert len(flowers) == 150
practice_fig.show(renderer="plotly_mimetype")

## Exercise 6. Export and check chart artifacts

Artifact checks complement visual inspection: a nonempty file alone might not be the expected
format. Keeping the JavaScript inside the HTML supports offline sharing at the cost of file size.

In [ ]:
with TemporaryDirectory() as folder:
    folder = Path(folder)
    fig, ax = plt.subplots(figsize=(6,3.5),layout="constrained")
    ax.bar(counts.index,counts.to_numpy())
    ax.set(title="Iris sample counts",xlabel="Species",ylabel="Flowers",ylim=(0,60))
    png = folder/"practice_counts.png"
    fig.savefig(png,dpi=100)
    plt.close(fig)
    practice_interactive = px.scatter(flowers,x="petal_length_cm",y="petal_width_cm",color="species",labels=labels)
    html = folder/"practice_scatter.html"
    practice_interactive.write_html(html,include_plotlyjs=True,auto_open=False)
    assert png.stat().st_size > 0 and png.read_bytes().startswith(b"\x89PNG\r\n\x1a\n")
    assert "Plotly.newPlot" in html.read_text(encoding="utf-8")
assert plt.get_fignums() == []

## Chapter project - An Iris exploration report

Write `create_visual_report(table)` returning a summary table, a two-panel Matplotlib Figure,
and an interactive Plotly Figure. Do not mutate its input. Require all four measurement columns
and species; reject empty data, missing/nonfinite/nonnumeric measurements, or missing/unknown species.
Use a normal loop for detailed validation if clearer than a comprehension.

The summary should contain each species' count and mean petal length. The static panels should
show a Seaborn petal-length box plot and a Matplotlib count bar chart. The interactive figure
should show petal length/width by species with hover and visible units. Keep the species order stable.
Save PNG and offline HTML in a temporary directory and verify them. Show the figures before closing.

**Check:** for flowers, summary counts `[50,50,50]`, total 150; two static axes; three interactive
traces totalling 150 points; the input is unchanged. An empty table, NaN measurement, infinity,
and unknown species must raise ValueError. A valid one-species subset may produce one trace.
Add two descriptive observations and one limitation. Avoid causal claims or a model-accuracy claim.

**Extend after Lesson 24:** build a held-out confusion matrix heatmap or residual scatter plot,
labelling actual/predicted axes explicitly. Those views evaluate a fitted model and must not be
confused with these exploratory feature charts.

### Reasoning

Validation belongs before chart construction, so a failed report cannot leave misleading partial
figures. Return figure objects to let the caller choose display and export. Reindex only categories
actually present, preserving the common order without inventing empty-group observations.

In [ ]:
from pandas.api.types import is_numeric_dtype

def create_visual_report(table):
    required = {*feature_names,"species"}
    if not isinstance(table,pd.DataFrame) or not required <= set(table.columns) or table.empty:
        raise ValueError("Expected nonempty Iris table with measurements and species")
    if not all(is_numeric_dtype(table[name]) for name in feature_names):
        raise ValueError("Measurements must be numeric")
    if table[feature_names].isna().any().any() or not np.isfinite(table[feature_names].to_numpy(dtype=float)).all():
        raise ValueError("Measurements must be finite and present")
    if table["species"].isna().any() or not table["species"].isin(species_order).all():
        raise ValueError("Unknown or missing species")
    order = [name for name in species_order if name in set(table["species"])]
    report_summary = table.groupby("species").agg(count=("petal_length_cm","size"),
                                                 mean_petal_length_cm=("petal_length_cm","mean")).reindex(order)
    fig, axes = plt.subplots(1,2,figsize=(10,3.5),layout="constrained")
    sns.boxplot(data=table,x="species",y="petal_length_cm",order=order,color="lightsteelblue",ax=axes[0])
    axes[0].set(title="Petal length spread",xlabel="Species",ylabel="Petal length (cm)")
    axes[1].bar(report_summary.index,report_summary["count"].to_numpy(),color="steelblue")
    axes[1].set(title="Included observations",xlabel="Species",ylabel="Flowers",ylim=(0,report_summary["count"].max()*1.2))
    chart = px.scatter(table,x="petal_length_cm",y="petal_width_cm",color="species",symbol="species",
        category_orders={"species":order},color_discrete_map=palette,labels=labels,
        title="Iris report: petal measurements")
    return report_summary, fig, chart

before = flowers.copy(deep=True)
report_summary, report_static, report_interactive = create_visual_report(flowers)
assert report_summary["count"].tolist() == [50,50,50]
assert report_summary["count"].sum() == 150 and len(report_static.axes) == 2
assert len(report_interactive.data) == 3
assert sum(len(trace.x) for trace in report_interactive.data) == 150
pd.testing.assert_frame_equal(flowers,before)
with TemporaryDirectory() as folder:
    folder = Path(folder)
    report_static.savefig(folder/"report.png",dpi=120)
    report_interactive.write_html(folder/"report.html",include_plotlyjs=True,auto_open=False)
    assert (folder/"report.png").read_bytes().startswith(b"\x89PNG\r\n\x1a\n")
    assert "Plotly.newPlot" in (folder/"report.html").read_text(encoding="utf-8")
plt.show()
plt.close(report_static)
report_interactive.show(renderer="plotly_mimetype")
bad_nan = flowers.copy(); bad_nan.loc[0,"petal_length_cm"] = np.nan
bad_inf = flowers.copy(); bad_inf.loc[0,"petal_length_cm"] = np.inf
bad_species = flowers.copy(); bad_species.loc[0,"species"] = "unknown"
for invalid in [flowers.iloc[:0],bad_nan,bad_inf,bad_species]:
    try:
        create_visual_report(invalid)
    except ValueError:
        pass
    else:
        raise AssertionError("Invalid table accepted")
one_summary, one_static, one_interactive = create_visual_report(flowers.loc[flowers["species"] == "setosa"])
assert one_summary["count"].tolist() == [50] and len(one_interactive.data) == 1
plt.close(one_static)
print("Observations: this teaching sample is balanced by species; setosa has shorter typical petals.")
print("Limitation: these selected observations do not establish population frequencies or causal effects.")